In [ ]:
!git clone https://github.com/knagaev/topic-modelling-attention.git
#%cd /content/drive/MyDrive/ARTM/topic-modelling-attention
#!git pull https://github.com/knagaev/topic-modelling-attention.git

In [1]:
import os, sys
sys.path.insert(0, '/kaggle/working/topic-modelling-attention/src/')

In [2]:
#!ls /kaggle/working/topic-modelling-attention
!mkdir /kaggle/working/results
!ls -l /kaggle/working/results

mkdir: cannot create directory ‘/kaggle/working/results’: File exists
total 0


In [ ]:
#!pip install /content/topic-modelling-attention/
#!pip install uv
!uv pip install -e /kaggle/working/topic-modelling-attention
!uv pip install --system --upgrade jax jaxlib
!uv pip install --upgrade "jax-cuda12-plugin==0.10.0"

In [ ]:
import nltk
nltk.download('stopwords')


In [3]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import ast
import json

import jax
import jax.numpy as jnp
from jax import Array

from nltk.stem import WordNetLemmatizer

from sklearn.datasets import fetch_20newsgroups
from sklearn.decomposition import LatentDirichletAllocation

import matplotlib.pyplot as plt
import seaborn as sns

from cartm import ContextTopicModel, AttentiveTopicModel
from cartm.preprocessing import (
    CorpusLoader,
    BatchedCorpusLoader,
    build_bow,
)
from cartm.metrics import (
    PerplexityMetric,
    NPMICoherenceMetric,
    SparsityMetric,
    TopicVarianceMetric,
)
from cartm.regularization import DecorrelationRegularization
from collections import Counter

In [4]:
RESULTS_DIR = '/kaggle/working/results/'

full_20newsgroups = fetch_20newsgroups(data_home='./data/', subset='all')
data = full_20newsgroups.data

#categories = ['alt.atheism', 'talk.religion.misc', 'comp.graphics', 'sci.space']
#data = fetch_20newsgroups(data_home='./data/', subset='train', categories=categories).data
#data = data[:100]

filter_mode = 'all'
if filter_mode == 'filtered':
    preprocessor = CorpusLoader(min_token_len=3, max_token_len=20, min_df=5, max_df=0.5)
if filter_mode == 'all':
    preprocessor = CorpusLoader(min_token_len=1, max_token_len=100, min_df=1, max_df=1.0, stopwords=set())
tokenized_data, document_bounds = preprocessor.fit_transform(data)
print(f'Total number of tokens in preprocessed corpus: {len(document_bounds)}')

loader = BatchedCorpusLoader(
    data=tokenized_data,
    doc_bounds=document_bounds,
    batch_size=10000,
)
print(f'Number of batches: {len(loader)}')

vocab_size = len(preprocessor.vocabulary)
print(f'vocab_size: {vocab_size}')

with open(RESULTS_DIR + filter_mode + "_phi_hist_vocab.json", "w") as f:
    json.dump(preprocessor.vocabulary, f, indent=4)

#bow = build_bow(tokenized_data, document_bounds, vocab_size)

#td = TopicVarianceMetric(top_k=25, tag="TD@25")
perplexity = PerplexityMetric()

decorr = DecorrelationRegularization(0.0, "wt")

model = AttentiveTopicModel(
    vocab_size=len(preprocessor.vocabulary),
    ctx_len=100,
    n_topics=100,
    gamma=0.6,
    metrics=[perplexity],
    regularizers=[], #[decorr],
    filter_mode=filter_mode
)

model.fit(
    loader,
    max_iter=50,
    verbose=2,
    seed=42,
)

np.save(RESULTS_DIR + filter_mode + "0.6_phi_hist.npy", model.phi_hist)

with open(RESULTS_DIR + filter_mode + "0.6_phi_hist_perplexity.txt", "w") as f:
     json.dump(perplexity.history, f)

Total number of tokens in preprocessed corpus: 5743601
Number of batches: 575
vocab_size: 115064
Iteration [1/50], phi update diff norm: 12.8867
  Metrics:
    PerplexityMetric: 1988.7859
Iteration [2/50], phi update diff norm: 15.4438
  Metrics:
    PerplexityMetric: 1936.6216
Iteration [3/50], phi update diff norm: 31.9788
  Metrics:
    PerplexityMetric: 1852.7997
Iteration [4/50], phi update diff norm: 65.6536
  Metrics:
    PerplexityMetric: 1693.3821
Iteration [5/50], phi update diff norm: 80.8353
  Metrics:
    PerplexityMetric: 1416.7040
Iteration [6/50], phi update diff norm: 69.5701
  Metrics:
    PerplexityMetric: 1131.5935
Iteration [7/50], phi update diff norm: 58.8798
  Metrics:
    PerplexityMetric: 947.3068
Iteration [8/50], phi update diff norm: 53.6822
  Metrics:
    PerplexityMetric: 825.5180
Iteration [9/50], phi update diff norm: 49.1703
  Metrics:
    PerplexityMetric: 739.7440
Iteration [10/50], phi update diff norm: 45.4153
  Metrics:
    PerplexityMetric: 680.4

In [6]:
import gzip
with gzip.GzipFile(RESULTS_DIR + filter_mode + '0.6_phi_hist.npy.gz', 'wb') as f:
    np.save(f, model.phi_hist)

In [ ]:
!ls -l /kaggle/working/results/
%cd /kaggle/working/
from IPython.display import FileLink
FileLink(r'results/all_phi_hist.npy.gz')

In [ ]:
from jax import jit
@jit
def entropy_renyi(matrix, power):
    entropies = jnp.sum(matrix ** power, axis=1)
    #return entropies
    return entropies / jnp.max(entropies)

In [ ]:
entropys = [(s, entropy_renyi(ilya_phi, power=s)) for s in np.linspace(1.1, 1.9, num=9)]
counts, bin_edges = jnp.histogram(entropys[0][1], range=(0, 1), bins=10)
counts

In [ ]:
def plot_distribution_advanced(data, n_bins=20, figsize=(12, 5),
                              plot_type='all', bandwidth=None):
    """
    Расширенная визуализация распределения.

    Args:
        data: 1D массив чисел в [0, 1]
        n_bins: количество бинов для гистограммы
        figsize: размер фигуры
        plot_type: 'hist' | 'kde' | 'ecdf' | 'all'
        bandwidth: ширина окна для KDE (по умолчанию автоматический выбор)

    Returns:
        dict с данными для каждого типа графика
    """
    #data = np.asarray(data)
    #n = len(data)

    results = {}

    # === ГИСТОГРАММА ===
    if plot_type in ['hist', 'all']:
        fig, ax = plt.subplots(figsize=figsize)
        counts, bin_edges, _ = ax.hist(
            data, bins=n_bins, range=(0, 1), density=True,
            color='skyblue', edgecolor='black', alpha=0.7, label='Гистограмма'
        )
        ax.set_xlabel('Значение', fontsize=11)
        ax.set_ylabel('Плотность', fontsize=11)
        ax.set_title('Гистограмма распределения', fontsize=13, pad=15)
        ax.grid(True, alpha=0.3, linestyle='--', axis='y')
        ax.set_xlim(0, 1)
        ax.legend()
        plt.tight_layout()
        plt.show()

        results['hist'] = {'counts': counts, 'bin_edges': bin_edges}

    # === KDE (ядерная оценка плотности) ===
    if plot_type in ['kde', 'all']:
        from scipy.stats import gaussian_kde

        # Автоматический bandwidth, если не задан
        if bandwidth is None:
            bandwidth = 1.06 * data.std() * n ** (-1/5)  # правило Сильвермана

        # Оценка плотности
        x_eval = np.linspace(0, 1, 200)
        kde = gaussian_kde(data, bw_method=bandwidth / data.std() if data.std() > 0 else 1.0)
        # Обрезаем выбросы KDE за пределами [0, 1]
        density = kde(x_eval)
        density = np.where((x_eval >= 0) & (x_eval <= 1), density, 0)

        plt.figure(figsize=figsize)
        plt.plot(x_eval, density, linewidth=2.5, color='darkorange', label='KDE')
        plt.fill_between(x_eval, density, alpha=0.2, color='darkorange')
        plt.xlabel('Значение', fontsize=11)
        plt.ylabel('Плотность', fontsize=11)
        plt.title('Ядерная оценка плотности (KDE)', fontsize=13, pad=15)
        plt.grid(True, alpha=0.3, linestyle='--')
        plt.xlim(0, 1)
        plt.legend()
        plt.tight_layout()
        plt.show()

        results['kde'] = {'x': x_eval, 'density': density}

    # === ECDF (эмпирическая функция распределения) ===
    if plot_type in ['ecdf', 'all']:
        n = len(data[0][1])

        ecdf = np.arange(1, n + 1) / n

        plt.figure(figsize=figsize)
        for i in range(len(data)):
        #for i in range(1):
            sorted_data = np.sort(np.asarray(data[i][1]))
            plt.plot(sorted_data, ecdf, linewidth=2, color='darkgreen', label='ECDF')
        plt.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Медиана')
        plt.axvline(x=np.median(sorted_data), color='red', linestyle=':', alpha=0.5)
        plt.xlabel('Значение', fontsize=11)
        plt.ylabel('Накопленная доля', fontsize=11)
        plt.title('Эмпирическая функция распределения (ECDF)', fontsize=13, pad=15)
        plt.grid(True, alpha=0.3, linestyle='--')
        plt.xlim(0, 1)
        plt.ylim(0, 1.05)
        plt.legend()
        plt.tight_layout()
        plt.show()

        results['ecdf'] = {'x': sorted_data, 'ecdf': ecdf}

    # Статистика
    #print(f"📊 Статистика данных (n={n})")
    #print(f"   Мин: {data.min():.4f}, Макс: {data.max():.4f}")
    #print(f"   Среднее: {data.mean():.4f}, Медиана: {np.median(data):.4f}")
    #print(f"   Стд. отклонение: {data.std():.4f}")
    #print(f"   Квантили: 25%={np.percentile(data, 25):.4f}, "
    #      f"75%={np.percentile(data, 75):.4f}")

    #return results

In [ ]:
plot_distribution_advanced(entropys, plot_type='ecdf')

In [ ]:
import json

with open('ilya_vocab.json', 'r') as file:
    reverse_vocab = json.load(file)

len(reverse_vocab)

In [ ]:
vocab = {value: key for key, value in reverse_vocab.items()}

In [ ]:
vocab['the']

In [ ]:
[p for p in ilya_phi[:, int(vocab['the'])]]

In [ ]:
plt.bar(range(20), ilya_phi[:, int(vocab['the'])])

In [ ]:
ilya_phi[ilya_phi > 0.99]

In [ ]:
mask = (ilya_phi > 0.99).any(axis=0)  # True для столбцов, содержащих хотя бы одну 1
result = ilya_phi[:, mask]          # выборка столбцов

In [ ]:
result.shape

In [ ]:
cols_idx = np.where(mask)[0]

In [ ]:
cols_idx

In [ ]:
[reverse_vocab[str(i)] for i in cols_idx]

In [ ]:
import json

with open('phi_hist_vocab.json', 'r') as file:
    vocab = json.load(file)
len(vocab)
reverse_vocab = {value: key for key, value in vocab.items()}

In [ ]:
phi_hist = np.load('phi_hist.npy').T
phi_hist.shape

In [ ]:
token = 'hundred'
token_hist = [p for p in phi_hist[:, int(vocab[token]), :]]
for i in range(len(token_hist)):
    plt.plot(token_hist[i])
plt.title(token)

In [ ]:
token = 'super'
token_hist = [p for p in phi_hist[:, int(vocab[token]), :]]
for i in range(len(token_hist)):
    plt.plot(token_hist[i])
plt.title(token)

In [ ]:
token = 'supercomputer'
token_hist = [p for p in phi_hist[:, int(vocab[token]), :]]
for i in range(len(token_hist)):
    plt.plot(token_hist[i])
plt.title(token)
print(phi_hist[:, int(vocab[token]), -1])

In [ ]:
token = 'catholicism'
token_hist = [p for p in phi_hist[:, int(vocab[token]), :]]
for i in range(len(token_hist)):
    plt.plot(token_hist[i])
plt.title(token)
print(phi_hist[:, int(vocab[token]), -1])

In [ ]:
token = 'the'
plt.bar(range(20), phi_hist[:, int(vocab[token]), 10])
plt.title(token)

In [ ]:
token = 'super'
plt.bar(range(20), phi_hist[:, int(vocab[token]), 10])
plt.title(token)

In [ ]:
token = 'supercomputer'
plt.bar(range(20), phi_hist[:, int(vocab[token]), 10])
plt.title(token)

In [ ]:
np.sum(phi_hist[:, :, -1], axis=1)
plt.bar(range(20), phi_hist[:, int(vocab['supercomputer']), 10])

In [ ]:
mask = (phi_hist[:, :, -1] > 0.99).any(axis=0)  # True для столбцов, содержащих хотя бы одну 1
result = phi_hist[:, mask, -1]         # выборка столбцов
result.shape

In [ ]:
cols_idx = np.where(mask)[0]
cols_idx

In [ ]:
plt.bar(range(20), np.sort(np.sum(result, axis=1)))
plt.title("Распределение якорных слов по темам")

In [ ]:
import json

with open('/content/all_phi_hist_vocab.json', 'r') as file:
    all_vocab = json.load(file)
    all_reverse_vocab = {value: key for key, value in all_vocab.items()}

with open('/content/filtered_phi_hist_vocab.json', 'r') as file:
    filtered_vocab = json.load(file)
    filtered_reverse_vocab = {value: key for key, value in filtered_vocab.items()}

print(f"{len(all_vocab)=}")
print(f"{len(filtered_vocab)=}")

In [ ]:
all_phi_hist = np.load('/content/all_phi_hist.npy').T
print(f"{all_phi_hist.shape=}")
filtered_phi_hist = np.load('/content/filtered_phi_hist.npy').T
print(f"{filtered_phi_hist.shape=}")

In [ ]:
token = 'good'
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 6))

all_token_hist = [p for p in all_phi_hist[:, int(all_vocab[token]), :]]
for i in range(len(all_token_hist)):
    ax1.plot(all_token_hist[i])
ax1.set_title('all ' + token)

filtered_token_hist = [p for p in filtered_phi_hist[:, int(filtered_vocab[token]), :]]
for i in range(len(filtered_token_hist)):
    ax2.plot(filtered_token_hist[i])
ax2.set_title('filtered ' + token)


In [ ]:
token = 'good'
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 6))

filtered_token_hist = [p for p in filtered_phi_hist[:, int(filtered_vocab[token]), :]]
for i in range(len(filtered_token_hist)):
    ax2.plot(filtered_token_hist[i])
ax2.set_title('filtered ' + token)


In [ ]:
token = 'bad'
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 6))

all_token_hist = [p for p in all_phi_hist[:, int(all_vocab[token]), :]]
for i in range(len(all_token_hist)):
    ax1.plot(all_token_hist[i])
ax1.set_title('all ' + token)

filtered_token_hist = [p for p in filtered_phi_hist[:, int(filtered_vocab[token]), :]]
for i in range(len(filtered_token_hist)):
    ax2.plot(filtered_token_hist[i])
ax2.set_title('filtered ' + token)


In [ ]:
with open('/content/20ng_anchors_tfidf.json', 'r') as file:
    anchors_tfidf = json.load(file)
motorcycles_anchors = anchors_tfidf['rec.motorcycles']
motorcycles_anchors_ids = [all_vocab[anchor] for anchor in motorcycles_anchors]
sum_topics_motorcycles = np.sum(all_phi_hist[:, motorcycles_anchors_ids, -1], axis=1)
sum_topics_motorcycles[sum_topics_motorcycles > 0.5]

In [ ]:
with open('/content/20ng_anchors_tfidf.json', 'r') as file:
    anchors_tfidf = json.load(file)
space_anchors = anchors_tfidf['sci.space']
space_anchors_ids = [all_vocab[anchor] for anchor in space_anchors]
sum_space_topics = np.sum(all_phi_hist[:, space_anchors_ids, -1], axis=1)
sum_space_topics[sum_space_topics > .5]

In [ ]:
with open('/content/20ng_anchors_tfidf.json', 'r') as file:
    anchors_tfidf = json.load(file)
mac_anchors = anchors_tfidf['comp.sys.mac.hardware']
mac_anchors_ids = [all_vocab[anchor] for anchor in mac_anchors]
sum_mac_topics = np.sum(all_phi_hist[:, mac_anchors_ids, -1], axis=1)
sum_mac_topics[sum_mac_topics > .5]

In [ ]:
mask = (all_phi_hist[:, :, -1] > 0.99).any(axis=0)  # True для столбцов, содержащих хотя бы одну 1
result = all_phi_hist[:, mask, -1]         # выборка столбцов
print(f"{result.shape=}")
print(f"{all_phi_hist.shape=}")

cols_idx = np.where(mask)[0]
print(f"{cols_idx=}")


In [ ]:
plt.bar(range(100), np.sort(np.sum(result, axis=1)))
plt.title("Распределение якорных слов по темам")

# Только filtered, потому что runtime отвалился

In [ ]:
filtered_phi_hist = np.load('/content/drive/MyDrive/ARTM/filtered_phi_hist.npy').T
print(f"{filtered_phi_hist.shape=}")

with open('/content/filtered_phi_hist_vocab.json', 'r') as file:
    filtered_vocab = json.load(file)
    filtered_reverse_vocab = {value: key for key, value in filtered_vocab.items()}

print(f"{len(filtered_vocab)=}")

In [ ]:
threshold = 1e-8

phi_bike = filtered_phi_hist[:, filtered_vocab['bike'], -1]
phi_bike[phi_bike < threshold] = 0
print(phi_bike)
bike_topic = np.where(phi_bike == 1)
bike_mask = (filtered_phi_hist[bike_topic, :, -1] == 1.0)
bike_anchors = [filtered_reverse_vocab[i] for i in np.flatnonzero(bike_mask)]
bike_anchors

In [ ]:
threshold = 1e-4

phi_space = filtered_phi_hist[:, filtered_vocab['rocketry'], -1] # распределение по темам
phi_space[phi_space < threshold] = 0
print(phi_space)
space_topic = np.where(phi_space > 1 - threshold) # тема, в которой это слово якорь
space_mask = (filtered_phi_hist[space_topic, :, -1] > 1 - threshold) # маска для слов, которые потенциально якорные в этой теме
#print(space_mask)
space_anchors = [filtered_reverse_vocab[i] for i in np.flatnonzero(space_mask)] # слова, которые потенциально якорные в этой теме
space_anchors

In [ ]:
threshold = 1e-7

phi_christian = filtered_phi_hist[:, filtered_vocab['archbishop'], -1] # распределение по темам
phi_christian[phi_christian < threshold] = 0
print(phi_christian)
christian_topic = np.where(phi_christian > 1 - threshold) # тема, в которой это слово якорь
christian_mask = (filtered_phi_hist[christian_topic, :, -1] > 1 - threshold) # маска для слов, которые потенциально якорные в этой теме
#print(christian_mask)
christian_anchors = [filtered_reverse_vocab[i] for i in np.flatnonzero(christian_mask)] # слова, которые потенциально якорные в этой теме
christian_anchors

In [ ]:
[k for k in filtered_vocab if k.startswith('schi')]

In [ ]:
docs_with_token = set()
for anchor in christian_anchors:
  #docs_with_token = [(i, t) for i, t in enumerate(data) if anchor in t.lower()]
  docs_with_token.update({i for i, t in enumerate(data) if anchor in t.lower()})

print(len(docs_with_token))


In [ ]:
counts = Counter([full_20newsgroups.target_names[full_20newsgroups.target[doc]] for doc in docs_with_token])
counts

In [ ]:
Надо взять слова с равномерным распределением по темам
Посмотреть на темы с самым большим количеством якорных слов

In [ ]:
data[7527: 7528]

In [ ]:
filtered_vocab['mmmmmmmmmm']

In [ ]:
filtered_reverse_vocab[12405]